# 03 — Model Training

Two-phase EfficientNet-B4 training with:
- Patient-level stratified splitting (no data leakage)
- **Professional class balancing**: weighted sampler + class-weighted loss
- Phase 1: head-only training
- Phase 2: fine-tuning with cosine warmup
- Mixed-precision (AMP) for speed
- Checkpointing on Quadratic Weighted Kappa

In [ ]:
import os, sys, importlib, shutil

# --- CONFIGURE DATA SOURCE ---
# "kaggle" = download from Kaggle (~1-3 min, recommended for Colab)
# "drive"  = copy from Google Drive (~15-30 min, fallback)
# "local"  = data already at expected path (no copy needed)
DATA_SOURCE = "kaggle"

IS_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in dir() else False

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/diabetic_retinopathy'
    %pip install -q timm albumentations

    LOCAL_DATA = '/content/dataset'
    if not os.path.isdir(os.path.join(LOCAL_DATA, 'resized_train_cropped')):
        if DATA_SOURCE == "kaggle":
            print('Upload your kaggle.json when prompted...')
            from google.colab import files
            uploaded = files.upload()  # user uploads kaggle.json
            os.makedirs('/root/.kaggle', exist_ok=True)
            shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
            os.chmod('/root/.kaggle/kaggle.json', 0o600)
            !kaggle datasets download -d tanlikesmath/diabetic-retinopathy-resized -p /content/ -q
            os.makedirs(LOCAL_DATA, exist_ok=True)
            import zipfile
            with zipfile.ZipFile('/content/diabetic-retinopathy-resized.zip', 'r') as z:
                z.extractall(LOCAL_DATA)
            os.remove('/content/diabetic-retinopathy-resized.zip')
            print('Kaggle download complete!')
        elif DATA_SOURCE == "drive":
            print('Copying dataset from Drive to local SSD (15-30 min)...')
            shutil.copytree(
                os.path.join(PROJECT_ROOT, 'dataset'), LOCAL_DATA,
                dirs_exist_ok=True,
            )
            print('Drive copy complete!')
    else:
        print('Dataset already on local SSD.')
else:
    PROJECT_ROOT = os.path.abspath('..')

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import src.config
importlib.reload(src.config)
from src import config

print(f'Images dir: {config.IMAGES_DIR}')
print(f'  exists: {os.path.isdir(config.IMAGES_DIR)}')
print(f'Outputs dir: {config.OUTPUTS_DIR}')

In [ ]:
import torch
import numpy as np

from src import config
from src.utils import seed_everything, get_device
from src.dataset import load_and_split_dataframe, compute_class_weights, create_dataloaders
from src.augmentations import get_train_transforms, get_valid_transforms
from src.model import build_model, count_parameters
from src.trainer import Trainer

seed_everything()
device = get_device()

## 1. Data Preparation with Balanced Splitting

In [ ]:
# Patient-level stratified split — both eyes of each patient stay together
train_df, valid_df, test_df = load_and_split_dataframe()

print(f'Train : {len(train_df):,} images  ({train_df["patient_id"].nunique():,} patients)')
print(f'Valid : {len(valid_df):,} images  ({valid_df["patient_id"].nunique():,} patients)')
print(f'Test  : {len(test_df):,} images  ({test_df["patient_id"].nunique():,} patients)')
print(f'\nTrain class distribution:')
print(train_df['level'].value_counts().sort_index())

In [ ]:
# Compute class weights for the loss function
class_weights = compute_class_weights(train_df['level'].values)
print('Class weights (balanced):')
for i, (name, w) in enumerate(zip(config.CLASS_NAMES, class_weights)):
    print(f'  {name:20s}: {w:.4f}')

In [ ]:
# Build DataLoaders with WeightedRandomSampler for balanced mini-batches
train_transforms = get_train_transforms()
valid_transforms = get_valid_transforms()

loaders = create_dataloaders(
    train_df=train_df,
    valid_df=valid_df,
    test_df=test_df,
    train_transform=train_transforms,
    valid_transform=valid_transforms,
    batch_size=config.PHASE1_BATCH_SIZE,
    use_sampler=True,  # WeightedRandomSampler for class balancing
)

print(f'\nTrain batches per epoch: {len(loaders["train"])}')
print(f'Valid batches          : {len(loaders["valid"])}')
print(f'Test  batches          : {len(loaders["test"])}')

## 2. Verify Balanced Sampling

The WeightedRandomSampler should produce roughly uniform class distribution in each batch, despite the 73%/5%/15%/2%/5% natural imbalance.

In [ ]:
# Verify that the sampler produces balanced batches
from collections import Counter

batch_labels = []
for i, (imgs, labels) in enumerate(loaders['train']):
    batch_labels.extend(labels.numpy().tolist())
    if i >= 19:  # check first 20 batches
        break

counter = Counter(batch_labels)
total = sum(counter.values())
print('Class distribution in first 20 training batches (with balanced sampler):')
for cls in sorted(counter.keys()):
    print(f'  Class {cls} ({config.CLASS_NAMES[cls]:20s}): {counter[cls]:4d} ({100*counter[cls]/total:.1f}%)')
print(f'  Total samples: {total}')

## 3. Build Model

In [ ]:
model = build_model(pretrained=True)
params = count_parameters(model)
print(f'Model: {config.MODEL_NAME}')
print(f'Total parameters : {params["total"]:,}')
print(f'Trainable        : {params["trainable"]:,} ({params["trainable_pct"]:.1f}%)')
print(f'Input size       : {config.IMG_SIZE}x{config.IMG_SIZE}')
print(f'Num classes      : {config.NUM_CLASSES}')

## 4. Train

Set `RESUME = True` to automatically continue from where training left off if
Colab disconnects or the runtime crashes.  Set to `False` to start fresh.

In [ ]:
RESUME = True   # flip to False to discard any saved state and train from scratch

trainer = Trainer(
    model=model,
    device=device,
    class_weights=class_weights,
)

if RESUME and Trainer.has_resume_state():
    print(f'Resume state found at {config.RESUME_STATE_PATH}')
else:
    print('Starting fresh training run.')

history = trainer.fit(
    train_loader=loaders['train'],
    valid_loader=loaders['valid'],
    resume=RESUME,
)

## 5. Training Curves

In [ ]:
from src.evaluate import plot_training_curves

plot_training_curves(
    history,
    save_path=os.path.join(config.FIGURES_DIR, 'training_curves.png'),
)

## 6. Quick Validation Check

In [ ]:
# Load the best checkpoint and run a quick check
trainer.load_best_checkpoint('phase2')

from src.evaluate import predict, compute_metrics

y_true, y_pred, y_proba = predict(model, loaders['valid'], device)
metrics = compute_metrics(y_true, y_pred, y_proba)

print(f'\nValidation Results:')
print(f'  QWK      : {metrics["qwk"]:.4f}')
print(f'  Accuracy : {metrics["accuracy"]:.4f}')
print(f'  ROC-AUC  : {metrics["roc_auc_weighted"]:.4f}')

In [ ]:
# Save test_df for the evaluation notebook
test_df.to_csv(os.path.join(config.DATA_DIR, 'test_split.csv'), index=False)
valid_df.to_csv(os.path.join(config.DATA_DIR, 'valid_split.csv'), index=False)
train_df.to_csv(os.path.join(config.DATA_DIR, 'train_split.csv'), index=False)
print('Split DataFrames saved for reproducible evaluation.')